# 🏥 ClinicAI v2.1 — NIH ChestX-ray14 | Kaggle Grandmaster Pipeline

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue)](https://python.org)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.x-red)](https://pytorch.org)
[![timm](https://img.shields.io/badge/timm-0.9%2B-green)](https://timm.fast.ai)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow)](LICENSE)

**Task:** Multi-label classification of 14 thoracic pathologies  
**Dataset:** NIH ChestX-ray14 (112 120 frontal chest X-rays)  
**Target:** Macro ROC-AUC > 0.84  
**Platform:** Kaggle (P100 / T4 / L4 GPU)  

---

## 📋 Table of Contents

| # | Section | Key Techniques |
|---|---------|----------------|
| 1 | [Environment Setup](#section-1) | Package installation, imports |
| 2 | [Reproducibility & Config](#section-2) | Seeds, determinism, global hyperparameters |
| 3 | [Dataset Analysis & EDA](#section-3) | Class distribution, co-occurrence, sample images |
| 4 | [Patient-Wise Data Split](#section-4) | Iterative multilabel stratification |
| 5 | [Data Pipeline](#section-5) | Albumentations: CLAHE, RRC, Mixup, CoarseDropout |
| 6 | [DataLoader](#section-6) | pin_memory, persistent_workers, prefetch |
| 7 | [Model Architecture](#section-7) | ConvNeXt / EfficientNetV2 / DenseNet / Swin |
| 8 | [Loss Functions](#section-8) | Weighted BCE, Focal Loss, Asymmetric Loss |
| 9 | [Optimiser](#section-9) | AdamW + layer-wise learning rates |
| 10 | [Scheduler](#section-10) | Cosine warmup / OneCycleLR |
| 11 | [Training Loop](#section-11) | AMP, gradient accumulation, EMA, early stopping |
| 12 | [Metrics](#section-12) | ROC-AUC, AP, P/R/F1, confusion matrix |
| 13 | [Threshold Optimisation](#section-13) | Per-class Youden's J statistic |
| 14 | [Test-Time Augmentation](#section-14) | Multi-scale + horizontal-flip TTA |
| 15 | [Explainability](#section-15) | GradCAM heatmap overlays |
| 16 | [Inference Pipeline](#section-16) | Single image, batch, submission CSV |
| 17 | [Error Analysis](#section-17) | FP / FN / hard / uncertain examples |
| 18 | [Training Tricks](#section-18) | Label smoothing, EMA, Mixup, CutMix |
| 19 | [Performance Optimisation](#section-19) | torch.compile, channels_last, cudnn |
| 20 | [Final Report](#section-20) | AUC table, thresholds, GPU stats |
| — | [▶ Run Full Pipeline](#run) | Execute end-to-end |

---
> **How to use:** Run all cells top-to-bottom on a Kaggle GPU kernel.  
> Set `CFG.WANDB = True` to stream metrics to Weights & Biases.

<a id="section-1"></a>

## 📦 Section 1 — Environment Setup & Package Installation

Install only the packages that are *not* pre-installed on the Kaggle GPU image.

| Package | Purpose |
|---------|---------|
| `timm` | State-of-the-art model zoo (ConvNeXt, EfficientNetV2, Swin …) |
| `albumentations` | Fast, GPU-friendly image augmentations |
| `grad-cam` | GradCAM explainability heatmaps |
| `iterstrat` | Iterative multilabel stratification for patient-wise splits |
| `wandb` | (Optional) experiment tracking |

> **Note:** Uncomment the `pip_install(...)` line below if any package is missing in your kernel.

In [24]:
# SECTION 1 — Environment Setup & Package Installation
# ===========================================================================
# On Kaggle most of these are pre-installed; we only install what is missing.
# We pin versions so the notebook stays reproducible when kernel images change.

import subprocess, sys

def pip_install(*packages):
    """Install packages silently; skip if already present."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages]
    )

# Uncomment on Kaggle if you need the latest versions:
# pip_install("timm>=0.9.12", "albumentations>=1.3.1", "grad-cam>=1.4.8",
#             "iterstrat>=0.1.7", "wandb>=0.16.0")

# ---------------------------------------------------------------------------
# Standard library
import os, gc, math, time, json, random, warnings, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Dict, List, Optional, Tuple, Union

# Data science
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                         # headless backend for Kaggle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

# Computer vision
import cv2

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T

# timm  — state-of-the-art model zoo
import timm

# Albumentations — fast GPU-friendly augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# scikit-learn
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve,
)
from sklearn.model_selection import train_test_split

# pytorch-grad-cam
try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
except ImportError:
    GRADCAM_AVAILABLE = False
    print("WARNING: GradCAM not available - Section 15 will be skipped.")

# iterstrat — iterative multilabel stratification
try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    ITERSTRAT_AVAILABLE = True
except ImportError:
    ITERSTRAT_AVAILABLE = False
    print("WARNING: iterstrat not available - falling back to random patient split.")

# Weights & Biases (optional)
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False

warnings.filterwarnings("ignore")
print("All imports successful.")


# ===========================================================================

All imports successful.


<a id="section-2"></a>

## 🔁 Section 2 — Reproducibility & Global Config

### Design Philosophy
- **`Config` class** — single source of truth for every hyperparameter and path.  
  Change a value here; all downstream modules pick it up automatically.
- **`set_seed()`** — fixes Python, NumPy, and PyTorch RNG for bit-exact reproducibility.
- **`CUDNN_BENCHMARK = True`** — auto-tunes cuDNN kernel selection for the fixed input size.

### Key Hyperparameters

| Param | Value | Rationale |
|-------|-------|-----------|
| `IMG_SIZE` | 320 | Sweet-spot for ConvNeXt-Base on P100 (memory vs accuracy) |
| `BATCH_SIZE` | 32 × 2 accum | Effective batch = 64; fits in 16 GB VRAM with AMP |
| `LR` | 3e-4 | Head LR; backbone gets 10× lower (3e-5) |
| `EMA_DECAY` | 0.9998 | Very slow EMA update — stable for long training runs |
| `ASL_GAMMA_NEG` | 4 | Aggressively down-weights easy negatives |

In [25]:
# SECTION 2 — Reproducibility & Global Config
# ===========================================================================

class Config:
    """
    Single source-of-truth for every hyper-parameter and path.
    Change values here; every module below reads from this object.
    """
    # ── Paths ──────────────────────────────────────────────────────────────
    DATA_DIR        = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
    IMAGE_DIR       = DATA_DIR / "images"
    META_CSV        = DATA_DIR / "Data_Entry_2017.csv"
    BBOX_CSV        = DATA_DIR / "BBox_List_2017.csv"
    SPLIT_FILE      = DATA_DIR / "train_val_list.txt"
    OUTPUT_DIR      = Path("/kaggle/working")
    CHECKPOINT_DIR  = OUTPUT_DIR / "checkpoints"
    GRAD_CAM_DIR    = OUTPUT_DIR / "gradcam"

    # ── Disease labels (14 pathologies) ────────────────────────────────────
    CLASSES = [
        "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
        "Mass", "Nodule", "Pneumonia", "Pneumothorax",
        "Consolidation", "Edema", "Emphysema", "Fibrosis",
        "Pleural_Thickening", "Hernia",
    ]
    NUM_CLASSES = len(CLASSES)          # 14

    # ── Image ───────────────────────────────────────────────────────────────
    IMG_SIZE        = 320              # ConvNeXt-Base sweet-spot on P100
    MEAN            = [0.485, 0.456, 0.406]
    STD             = [0.229, 0.224, 0.225]

    # ── Training ────────────────────────────────────────────────────────────
    EPOCHS          = 30
    BATCH_SIZE      = 32               # effective batch = BATCH * ACCUM_STEPS
    ACCUM_STEPS     = 2                # gradient accumulation steps
    NUM_WORKERS     = 4
    PIN_MEMORY      = True
    PERSISTENT_WORKERS = True
    PREFETCH_FACTOR = 2

    # ── Optimiser ───────────────────────────────────────────────────────────
    LR              = 3e-4
    LR_MIN          = 1e-6
    WEIGHT_DECAY    = 1e-2
    GRAD_CLIP       = 1.0

    # ── Scheduler ───────────────────────────────────────────────────────────
    SCHEDULER       = "cosine_warmup"  # "cosine_warmup" | "onecycle"
    WARMUP_EPOCHS   = 3

    # ── Model ───────────────────────────────────────────────────────────────
    BACKBONE        = "convnext_base"  # default backbone
    PRETRAINED      = True
    DROP_RATE       = 0.2
    DROP_PATH_RATE  = 0.1

    # ── Loss ────────────────────────────────────────────────────────────────
    LOSS            = "asymmetric"     # "bce" | "focal" | "asymmetric"
    LABEL_SMOOTH    = 0.05

    # ── Asymmetric Loss ─────────────────────────────────────────────────────
    ASL_GAMMA_NEG   = 4
    ASL_GAMMA_POS   = 1
    ASL_CLIP        = 0.05

    # ── Focal Loss ──────────────────────────────────────────────────────────
    FOCAL_GAMMA     = 2.0
    FOCAL_ALPHA     = 0.25

    # ── EMA ─────────────────────────────────────────────────────────────────
    EMA_DECAY       = 0.9998

    # ── Augmentation ────────────────────────────────────────────────────────
    USE_MIXUP       = True
    MIXUP_ALPHA     = 0.4
    USE_CUTMIX      = True
    CUTMIX_ALPHA    = 1.0

    # ── Early Stopping ──────────────────────────────────────────────────────
    PATIENCE        = 7
    MIN_DELTA       = 1e-4

    # ── TTA ─────────────────────────────────────────────────────────────────
    TTA_SCALES      = [288, 320, 352]

    # ── Misc ────────────────────────────────────────────────────────────────
    SEED            = 42
    AMP             = True             # Automatic Mixed Precision
    CHANNELS_LAST   = True             # memory-format optimisation
    COMPILE         = False            # torch.compile (PyTorch >= 2.0)
    CUDNN_BENCHMARK = True
    WANDB           = False            # set True + fill project name to enable

    # ── WandB ───────────────────────────────────────────────────────────────
    WANDB_PROJECT   = "clinicai-chestxray14"
    WANDB_RUN_NAME  = f"convnext_base_{datetime.now().strftime('%Y%m%d_%H%M')}"


CFG = Config()

# ── Create output directories ───────────────────────────────────────────────
CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CFG.GRAD_CAM_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42) -> None:
    """
    Make training fully deterministic.
    Note: determinism may slightly reduce throughput; acceptable on Kaggle.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # reproducible conv results
    torch.backends.cudnn.benchmark     = CFG.CUDNN_BENCHMARK  # auto-tune kernels
    os.environ["PYTHONHASHSEED"]       = str(seed)

set_seed(CFG.SEED)

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ===========================================================================

Device : cuda
  GPU    : Tesla T4
  VRAM   : 15.6 GB


<a id="section-3"></a>

## 📊 Section 3 — Dataset Analysis & EDA

`DataAnalyser` produces **6 publication-ready figures** saved to `/kaggle/working/`:

| Figure | Shows |
|--------|-------|
| `class_distribution.png` | Bar chart of per-class prevalence (huge imbalance!) |
| `cooccurrence_matrix.png` | Conditional probability P(col \| row) — disease co-morbidity |
| `patient_distribution.png` | Images-per-patient histogram + patient split pie |
| `view_position.png` | PA vs AP vs LL breakdown |
| `sample_images.png` | One representative X-ray per disease class |

**Printed table** shows `neg:pos` ratio and prevalence % for every class.

> Hernia has only ~227 images with a 500:1 imbalance — the hardest class.

In [26]:
# SECTION 3 — Dataset Analysis & EDA
# ===========================================================================

class DataAnalyser:
    """
    Encapsulates all exploratory data analysis for the NIH ChestX-ray14 dataset.
    Produces publication-ready figures saved to CFG.OUTPUT_DIR.
    """

    def __init__(self, df: pd.DataFrame, cfg: Config):
        self.df  = df
        self.cfg = cfg
        sns.set_theme(style="darkgrid", palette="muted")

    # ------------------------------------------------------------------
    def _parse_labels(self) -> pd.DataFrame:
        """
        Expand pipe-delimited Finding Labels into binary columns.
        Returns the original dataframe with 14 additional binary columns.
        """
        df = self.df.copy()
        for cls in self.cfg.CLASSES:
            df[cls] = df["Finding Labels"].apply(
                lambda x: 1 if cls in x.split("|") else 0
            )
        return df

    # ------------------------------------------------------------------
    def run(self) -> pd.DataFrame:
        """Execute all EDA steps and return the augmented dataframe."""
        df = self._parse_labels()
        self._plot_class_distribution(df)
        self._plot_cooccurrence(df)
        self._plot_patient_distribution(df)
        self._plot_view_position(df)
        self._plot_sample_images(df)
        self._print_imbalance_stats(df)
        return df

    # ------------------------------------------------------------------
    def _plot_class_distribution(self, df: pd.DataFrame) -> None:
        counts = df[self.cfg.CLASSES].sum().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(14, 5))
        bars = ax.bar(counts.index, counts.values,
                      color=plt.cm.viridis(np.linspace(0.2, 0.9, len(counts))))
        ax.set_title("Class Distribution — NIH ChestX-ray14", fontsize=15, fontweight="bold")
        ax.set_ylabel("Number of Images")
        ax.set_xticklabels(counts.index, rotation=45, ha="right")
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
                    f"{val:,}", ha="center", va="bottom", fontsize=8)
        plt.tight_layout()
        plt.savefig(self.cfg.OUTPUT_DIR / "class_distribution.png", dpi=150)
        plt.close()
        print("Saved: class_distribution.png")

    # ------------------------------------------------------------------
    def _plot_cooccurrence(self, df: pd.DataFrame) -> None:
        label_mat = df[self.cfg.CLASSES].values
        co = (label_mat.T @ label_mat).astype(float)
        diag = np.diag(co)
        with np.errstate(divide="ignore", invalid="ignore"):
            co_norm = co / diag[:, None]
        np.fill_diagonal(co_norm, 0.0)

        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(co_norm, xticklabels=self.cfg.CLASSES, yticklabels=self.cfg.CLASSES,
                    annot=True, fmt=".2f", cmap="YlOrRd", ax=ax, linewidths=0.3)
        ax.set_title("Disease Co-occurrence Matrix  (P(col | row))",
                     fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(self.cfg.OUTPUT_DIR / "cooccurrence_matrix.png", dpi=150)
        plt.close()
        print("Saved: cooccurrence_matrix.png")

    # ------------------------------------------------------------------
    def _plot_patient_distribution(self, df: pd.DataFrame) -> None:
        imgs_per_patient = df.groupby("Patient ID").size()
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        axes[0].hist(imgs_per_patient.values, bins=40, color="#4C72B0", edgecolor="white")
        axes[0].set_title("Images per Patient")
        axes[0].set_xlabel("Number of Images")
        axes[0].set_ylabel("Number of Patients")

        n_patients = len(df["Patient ID"].unique())
        half = n_patients // 2
        axes[1].pie(
            [half, n_patients - half],
            labels=["First Half Patients", "Second Half Patients"],
            autopct="%1.1f%%", colors=["#4C72B0", "#DD8452"]
        )
        axes[1].set_title("Patient Data Distribution")
        plt.suptitle("Patient-Level Analysis", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(self.cfg.OUTPUT_DIR / "patient_distribution.png", dpi=150)
        plt.close()
        print("Saved: patient_distribution.png")

    # ------------------------------------------------------------------
    def _plot_view_position(self, df: pd.DataFrame) -> None:
        vc = df["View Position"].value_counts()
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.pie(vc.values, labels=vc.index, autopct="%1.1f%%",
               colors=["#4C72B0", "#DD8452", "#55A868"])
        ax.set_title("View Position Distribution", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.savefig(self.cfg.OUTPUT_DIR / "view_position.png", dpi=150)
        plt.close()
        print("Saved: view_position.png")

    # ------------------------------------------------------------------
    def _plot_sample_images(self, df: pd.DataFrame) -> None:
        """Display one representative image for each disease class."""
        fig, axes = plt.subplots(2, 7, figsize=(22, 7))
        axes = axes.flatten()
        for i, cls in enumerate(self.cfg.CLASSES):
            subset = df[df[cls] == 1]["Image Index"].values
            if len(subset) == 0:
                axes[i].axis("off")
                continue
            img_path = self.cfg.IMAGE_DIR / random.choice(subset)
            if img_path.exists():
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                axes[i].imshow(img, cmap="gray")
            else:
                axes[i].axis("off")
            axes[i].set_title(cls, fontsize=9, fontweight="bold")
            axes[i].axis("off")
        plt.suptitle("Representative Images — One per Disease", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.savefig(self.cfg.OUTPUT_DIR / "sample_images.png", dpi=150)
        plt.close()
        print("Saved: sample_images.png")

    # ------------------------------------------------------------------
    def _print_imbalance_stats(self, df: pd.DataFrame) -> None:
        total   = len(df)
        pos     = df[self.cfg.CLASSES].sum()
        neg     = total - pos
        ratio   = (neg / pos).rename("neg:pos ratio")
        stats   = pd.concat([pos.rename("# Positive"), neg.rename("# Negative"),
                              ratio, (pos / total * 100).rename("Prevalence %")], axis=1)
        stats   = stats.sort_values("# Positive", ascending=False)
        print("\nClass Imbalance Statistics")
        print("=" * 65)
        print(stats.to_string())
        print("=" * 65)


# ===========================================================================

<a id="section-4"></a>

## ✂️ Section 4 — Patient-Wise Data Split

### Why Patient-Wise?
The NIH dataset contains **multiple images per patient**.  A naive random split
would leak the same patient into both train and validation, inflating AUC by
several points — a common mistake in chest X-ray literature.

### Strategy
1. Aggregate labels per patient (`OR` across all their images).
2. Run **`MultilabelStratifiedShuffleSplit`** (iterstrat) so each class has
   ~equal prevalence in train and val.
3. Fall back to a plain random patient split if iterstrat is unavailable.

```
Total patients ≈ 30 805
Train patients ≈ 26 184  (85%)
Val   patients ≈  4 621  (15%)
```

**Assertion:** The code verifies zero patient overlap before proceeding.

In [27]:
# SECTION 4 — Patient-Wise Data Split
# ===========================================================================

class DataSplitter:
    """
    Performs a STRICT patient-wise train/val split.
    No patient's images may appear in both sets.

    Strategy
    --------
    1. Aggregate per-patient label vectors (OR over all their images).
    2. Use MultilabelStratifiedShuffleSplit (iterstrat) so the validation
       set has similar class prevalence to the training set.
    3. Fall back to random patient split if iterstrat is unavailable.
    """

    def __init__(self, df: pd.DataFrame, cfg: Config, val_ratio: float = 0.15):
        self.df        = df
        self.cfg       = cfg
        self.val_ratio = val_ratio

    # ------------------------------------------------------------------
    def split(self) -> Tuple[pd.DataFrame, pd.DataFrame]:
        patient_ids = self.df["Patient ID"].unique()
        print(f"Total patients : {len(patient_ids):,}")

        # Per-patient aggregated label matrix
        patient_labels = (
            self.df.groupby("Patient ID")[self.cfg.CLASSES]
            .max()           # OR — patient is "positive" if any image shows it
            .reset_index()
        )

        if ITERSTRAT_AVAILABLE:
            splitter = MultilabelStratifiedShuffleSplit(
                n_splits=1, test_size=self.val_ratio, random_state=self.cfg.SEED
            )
            X = patient_labels["Patient ID"].values.reshape(-1, 1)
            y = patient_labels[self.cfg.CLASSES].values
            train_idx, val_idx = next(splitter.split(X, y))
            train_patients = patient_labels.iloc[train_idx]["Patient ID"].values
            val_patients   = patient_labels.iloc[val_idx]["Patient ID"].values
            print("Used iterative multilabel stratification.")
        else:
            train_patients, val_patients = train_test_split(
                patient_ids, test_size=self.val_ratio,
                random_state=self.cfg.SEED, shuffle=True
            )
            print("Used random patient split (iterstrat unavailable).")

        train_df = self.df[self.df["Patient ID"].isin(train_patients)].reset_index(drop=True)
        val_df   = self.df[self.df["Patient ID"].isin(val_patients)].reset_index(drop=True)

        # Sanity check — no patient overlap
        assert len(set(train_df["Patient ID"]) & set(val_df["Patient ID"])) == 0, \
            "ERROR: Patient leakage detected!"

        print(f"Train images : {len(train_df):,}  ({len(train_patients):,} patients)")
        print(f"Val   images : {len(val_df):,}  ({len(val_patients):,} patients)")
        return train_df, val_df


# ===========================================================================

<a id="section-5"></a>

## 🔬 Section 5 — Data Pipeline (Albumentations)

### Training Augmentations

| Transform | p | Rationale |
|-----------|---|-----------|
| `Resize(352, 352)` | 1.0 | Slightly oversized before crop |
| `RandomResizedCrop(320)` | 1.0 | Scale/aspect-ratio jitter |
| `HorizontalFlip` | 0.5 | Chest X-rays are horizontally symmetric |
| `ShiftScaleRotate` | 0.5 | Small geometric deformation |
| `CLAHE` | 0.5 | **Key for X-rays** — improves local contrast |
| `RandomBrightnessContrast` | 0.5 | Intensity variation |
| `GaussNoise` | 0.3 | Simulates sensor noise |
| `GaussianBlur` | 0.1 | Mild blur regularisation |
| `CoarseDropout` | 0.2 | Occlusion regularisation |
| `Normalize` | 1.0 | ImageNet mean/std (all backbones are ImageNet-pretrained) |

### Validation: Resize → Normalize only (no randomness).

In [28]:
def build_transforms(mode: str, cfg: Config) -> A.Compose:

    mean, std = cfg.MEAN, cfg.STD

    if mode == "train":
        return A.Compose([

            # Spatial
            A.Resize(cfg.IMG_SIZE + 32, cfg.IMG_SIZE + 32),

            A.RandomResizedCrop(
                size=(cfg.IMG_SIZE, cfg.IMG_SIZE),
                scale=(0.75, 1.0),
                ratio=(0.85, 1.15),
                p=1.0,
            ),

            A.HorizontalFlip(p=0.5),

            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=10,
                border_mode=cv2.BORDER_REFLECT_101,
                p=0.5,
            ),

            # Intensity
            A.CLAHE(
                clip_limit=2.0,
                tile_grid_size=(8, 8),
                p=0.5,
            ),

            A.RandomBrightnessContrast(
                brightness_limit=0.15,
                contrast_limit=0.15,
                p=0.5,
            ),

            A.GaussNoise(
                std_range=(0.02, 0.08),
                p=0.3,
            ),

            A.GaussianBlur(
                blur_limit=(3, 5),
                p=0.1,
            ),

            # Regularization
            A.CoarseDropout(
                num_holes_range=(1, 8),
                hole_height_range=(0.02, 0.06),
                hole_width_range=(0.02, 0.06),
                fill=0,
                p=0.2,
            ),

            # Normalize
            A.Normalize(
                mean=mean,
                std=std,
            ),

            ToTensorV2(),
        ])

    elif mode in ("val", "test"):

        return A.Compose([

            A.Resize(cfg.IMG_SIZE, cfg.IMG_SIZE),

            A.Normalize(
                mean=mean,
                std=std,
            ),

            ToTensorV2(),
        ])

    else:
        raise ValueError(f"Unknown mode: {mode}")

<a id="section-6"></a>

## ⚡ Section 6 — Dataset Class & DataLoader

### `ChestXray14Dataset`
- Loads grayscale PNGs → converts to 3-channel RGB (all ImageNet backbones expect RGB).
- Pre-computes the label matrix as `float32` numpy array at construction time.
- Returns `(image_tensor, label_tensor)` pairs.

### DataLoader Optimisations for Kaggle

| Option | Value | Effect |
|--------|-------|--------|
| `num_workers` | 4 | Parallel CPU data loading |
| `pin_memory` | True | Page-locked RAM → faster CPU→GPU transfer |
| `persistent_workers` | True | Workers survive across epochs (saves fork overhead) |
| `prefetch_factor` | 2 | Each worker pre-fetches 2 batches ahead |
| `drop_last` | True (train) | Avoids tiny last batch causing BatchNorm instability |

In [29]:
# ==========================================================
# Build image map once
# ==========================================================

def build_image_map(root="/kaggle/input"):
    image_map = {}

    for img in Path(root).rglob("*.png"):
        image_map[img.name] = img

    print(f"Found {len(image_map)} images.")
    return image_map


IMAGE_MAP = build_image_map()

Found 112125 images.


In [30]:
# ==========================================================
# Dataset
# ==========================================================

class ChestXray14Dataset(Dataset):

    def __init__(self,
                 df,
                 image_dir,      # يبقى موجوداً للتوافق مع الكود القديم
                 transform,
                 cfg):

        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.cfg = cfg

        # نتجاهل image_dir ونستخدم الخريطة
        self.image_map = IMAGE_MAP

        self.labels = df[cfg.CLASSES].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        filename = row["Image Index"]

        img_path = self.image_map.get(filename)

        if img_path is None:

            img = np.zeros(
                (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE),
                dtype=np.uint8,
            )

        else:

            img = cv2.imread(
                str(img_path),
                cv2.IMREAD_GRAYSCALE,
            )

            if img is None:

                img = np.zeros(
                    (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE),
                    dtype=np.uint8,
                )

        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        img = self.transform(image=img)["image"]

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32,
        )

        return img, label

    def get_pos_weight(self):

        pos = self.labels.sum(axis=0)
        neg = len(self.labels) - pos

        weight = np.where(
            pos > 0,
            neg / pos,
            100.0,
        )

        return torch.tensor(
            weight,
            dtype=torch.float32,
        )

<a id="section-7"></a>

## 🧠 Section 7 — Model Architecture

### Modular Backbone Framework
Switch backbone by changing **one line**: `CFG.BACKBONE = "..."`.

| Backbone | Params | Notes |
|----------|--------|-------|
| `convnext_base` ✅ | 89 M | **Default.** Best accuracy on ChestX-ray14 |
| `tf_efficientnetv2_m` | 54 M | Memory-efficient; faster than ConvNeXt |
| `densenet121` | 8 M | Original CheXNet backbone; lightweight baseline |
| `swin_base_patch4_window7_224` | 88 M | Vision Transformer; strong on global patterns |
| `convnext_small` | 50 M | Faster ConvNeXt variant |

### Classification Head
```
backbone (pretrained) → GAP
    → LayerNorm(num_features)
    → Dropout(0.2)
    → Linear(num_features → 512)
    → GELU
    → Dropout(0.1)
    → Linear(512 → 14)       # raw logits, no sigmoid
```
> **No sigmoid in forward()** — loss functions operate on logits for numerical stability.

In [31]:
# SECTION 7 — Model Architecture
# ===========================================================================

class ChestXrayModel(nn.Module):
    """
    Modular multi-label classification head on top of any timm backbone.

    Supported backbones (pass as cfg.BACKBONE)
    -------------------------------------------
    * "convnext_base"                    — ConvNeXt-Base   (default)
    * "tf_efficientnetv2_m"              — EfficientNetV2-M
    * "densenet121"                      — DenseNet-121  (CheXNet baseline)
    * "swin_base_patch4_window7_224"     — Swin-B Transformer
    * "convnext_small"                   — ConvNeXt-Small  (faster)
    * "tf_efficientnetv2_s"              — EfficientNetV2-S (faster)

    Architecture
    ------------
    backbone -> Global Average Pool -> LayerNorm -> Dropout ->
    Linear(512) -> GELU -> Dropout -> Linear(NUM_CLASSES)

    Raw logits are output (no sigmoid). Use BCEWithLogitsLoss,
    FocalLoss, or AsymmetricLoss during training for numerical stability.
    """

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        # ── Build backbone via timm ──────────────────────────────────────
        self.backbone = timm.create_model(
            cfg.BACKBONE,
            pretrained      = cfg.PRETRAINED,
            num_classes     = 0,           # remove classification head
            global_pool     = "avg",       # global average pooling
            drop_rate       = cfg.DROP_RATE,
            drop_path_rate  = cfg.DROP_PATH_RATE,
        )
        num_features = self.backbone.num_features
        print(f"Backbone : {cfg.BACKBONE}  |  Features : {num_features}")

        # ── Classification head ─────────────────────────────────────────
        self.head = nn.Sequential(
            nn.LayerNorm(num_features),
            nn.Dropout(p=cfg.DROP_RATE),
            nn.Linear(num_features, 512),
            nn.GELU(),
            nn.Dropout(p=cfg.DROP_RATE / 2),
            nn.Linear(512, cfg.NUM_CLASSES),
        )
        self._init_head()

    # ------------------------------------------------------------------
    def _init_head(self) -> None:
        """Xavier initialisation on head linear layers."""
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    # ------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : [B, 3, H, W]  input images (normalised)
        Returns [B, NUM_CLASSES] raw logits (no activation).
        """
        features = self.backbone(x)     # [B, num_features]
        logits   = self.head(features)  # [B, NUM_CLASSES]
        return logits

    # ------------------------------------------------------------------
    def get_gradcam_layer(self) -> nn.Module:
        """Return the last convolutional layer suitable for GradCAM."""
        if "convnext" in self.cfg.BACKBONE:
            return self.backbone.stages[-1].blocks[-1]
        elif "densenet" in self.cfg.BACKBONE:
            return self.backbone.features.denseblock4
        elif "efficientnet" in self.cfg.BACKBONE:
            return self.backbone.blocks[-1]
        elif "swin" in self.cfg.BACKBONE:
            return self.backbone.layers[-1].blocks[-1]
        else:
            # Generic fallback: last module with weight
            for module in reversed(list(self.backbone.modules())):
                if hasattr(module, "weight"):
                    return module
            return self.backbone

    # ------------------------------------------------------------------
    @torch.no_grad()
    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Return sigmoid probabilities for inference."""
        self.eval()
        logits = self(x)
        return torch.sigmoid(logits)

    # ------------------------------------------------------------------
    def param_count(self) -> int:
        return sum(p.numel() for p in self.parameters())

    def trainable_param_count(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ===========================================================================

<a id="section-8"></a>

## 📉 Section 8 — Loss Functions

Three losses are implemented and benchmarked. **Asymmetric Loss is the default.**

### Weighted BCE
Standard BCEWithLogitsLoss with per-class `pos_weight = neg/pos`.
+ Label smoothing replaces hard {0,1} targets with soft {ε/2, 1−ε/2}.

### Focal Loss (Lin et al., 2017)
`FL(p) = −α(1−p)^γ log(p)` — down-weights easy examples, focuses on hard ones.

### Asymmetric Loss ✅ (Ridnik et al., 2021)
Uses **different γ** for positives and negatives:
- `γ_pos = 1` → mild down-weighting of easy positives
- `γ_neg = 4` → aggressive down-weighting of easy negatives
- **Probability shifting** (clip=0.05): removes very confident negatives from gradient

> ASL consistently outperforms BCE and Focal on multi-label medical datasets.

In [32]:
# SECTION 8 — Loss Functions
# ===========================================================================

class WeightedBCELoss(nn.Module):
    """
    Weighted Binary Cross Entropy with optional label smoothing.

    Label smoothing replaces hard targets {0, 1} with soft targets
    {eps/2, 1-eps/2}, reducing over-confidence and improving calibration.
    """
    def __init__(self, pos_weight: torch.Tensor, label_smooth: float = 0.05):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
        self.smooth = label_smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.smooth > 0:
            targets = targets * (1 - self.smooth) + 0.5 * self.smooth
        return F.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight, reduction="mean"
        )


class FocalLoss(nn.Module):
    """
    Sigmoid Focal Loss (Lin et al., 2017 — RetinaNet).

    Reduces the contribution of easy negatives so the model focuses
    on hard, mis-classified examples — extremely useful for rare diseases.

    FL(p) = -alpha * (1-p)^gamma * log(p)   for positives
    FL(p) = -(1-alpha) * p^gamma * log(1-p) for negatives
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25,
                 label_smooth: float = 0.05):
        super().__init__()
        self.gamma  = gamma
        self.alpha  = alpha
        self.smooth = label_smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.smooth > 0:
            targets = targets * (1 - self.smooth) + 0.5 * self.smooth
        probs = torch.sigmoid(logits)
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt    = probs * targets + (1 - probs) * (1 - targets)
        alpha = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal = alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()


class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss (Ridnik et al., 2021 — ASL).

    Key insight: in multi-label problems, false negatives (missing a disease)
    are far more costly than false positives.  ASL uses different focusing
    parameters for positives and negatives:
      - gamma_pos (small, e.g. 1): mild downweighting of easy positives
      - gamma_neg (large, e.g. 4): aggressive downweighting of easy negatives
    Probability shifting clip: squeezes p_neg to [0, 1-m] to remove
    very confident negatives from the gradient entirely.

    State-of-the-art for multi-label medical imaging (consistently beats BCE
    and Focal Loss on CheXpert, NIH ChestX-ray14, etc.).
    """
    def __init__(self, gamma_neg: int = 4, gamma_pos: int = 1,
                 clip: float = 0.05, label_smooth: float = 0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.smooth    = label_smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.smooth > 0:
            targets = targets * (1 - self.smooth) + 0.5 * self.smooth

        xs_pos = torch.sigmoid(logits)
        xs_neg = 1 - xs_pos

        # Probability shifting: clip easy negatives
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1.0)

        los_pos = targets       * torch.log(xs_pos.clamp(min=1e-8))
        los_neg = (1 - targets) * torch.log(xs_neg.clamp(min=1e-8))
        loss    = los_pos + los_neg

        # Asymmetric focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            pt0             = xs_pos * targets
            pt1             = xs_neg * (1 - targets)
            pt              = pt0 + pt1
            one_sided_gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
            one_sided_w     = torch.pow(1 - pt, one_sided_gamma)
            loss           *= one_sided_w

        return -loss.sum() / logits.size(0)


def build_loss(cfg: Config, pos_weight: Optional[torch.Tensor] = None) -> nn.Module:
    """Factory: instantiate and return the configured loss function."""
    if cfg.LOSS == "bce":
        assert pos_weight is not None, "pos_weight required for WeightedBCELoss"
        return WeightedBCELoss(pos_weight.to(DEVICE), cfg.LABEL_SMOOTH)
    elif cfg.LOSS == "focal":
        return FocalLoss(cfg.FOCAL_GAMMA, cfg.FOCAL_ALPHA, cfg.LABEL_SMOOTH)
    elif cfg.LOSS == "asymmetric":
        return AsymmetricLoss(cfg.ASL_GAMMA_NEG, cfg.ASL_GAMMA_POS,
                              cfg.ASL_CLIP, cfg.LABEL_SMOOTH)
    else:
        raise ValueError(f"Unknown loss: {cfg.LOSS}")


# ===========================================================================

<a id="section-9"></a>

## ⚙️ Section 9 — Optimiser (AdamW)

**AdamW** with decoupled weight decay (Loshchilov & Hutter, 2019).

### Layer-wise Learning Rate Decay (LLRD)
| Parameter Group | Learning Rate | Rationale |
|-----------------|---------------|-----------|
| Backbone | `LR × 0.1 = 3e-5` | Pre-trained weights need gentle updates |
| Head | `LR × 1.0 = 3e-4` | Randomly initialised — needs aggressive training |

This pattern is standard in BERT fine-tuning and transfers well to vision.

In [33]:
# SECTION 9 — Optimiser
# ===========================================================================

def build_optimiser(model: nn.Module, cfg: Config) -> torch.optim.Optimizer:
    """
    AdamW with decoupled weight decay.

    Layer-wise learning rates (LLRD)
    ---------------------------------
    - backbone : lr * 0.1   (fine-tune conservatively; weights are pre-trained)
    - head     : lr * 1.0   (train aggressively; weights are randomly initialised)

    This is a standard technique from BERT fine-tuning applied to vision.
    """
    backbone_params = list(model.backbone.parameters())
    head_params     = list(model.head.parameters())

    param_groups = [
        {"params": backbone_params, "lr": cfg.LR * 0.1, "name": "backbone"},
        {"params": head_params,     "lr": cfg.LR,       "name": "head"},
    ]
    return torch.optim.AdamW(param_groups, weight_decay=cfg.WEIGHT_DECAY)


# ===========================================================================

<a id="section-10"></a>

## 📈 Section 10 — Learning Rate Scheduler

Two scheduler options are available via `CFG.SCHEDULER`.

### Cosine Warmup ✅ (default)
```
Epoch 0 → WARMUP_EPOCHS : LR increases linearly 0 → base_lr
Epoch WARMUP → EPOCHS   : LR decays base_lr → LR_MIN  (cosine curve)
```
Warmup prevents head weight divergence in the first epochs (large gradients).

### OneCycleLR (alternative)
Super-convergence policy from Smith & Topin (2018).  
Max LR → step-up → step-down in a single cycle.  
Set `CFG.SCHEDULER = "onecycle"` to activate.

In [34]:
# SECTION 10 — Scheduler
# ===========================================================================

class CosineWarmupScheduler(torch.optim.lr_scheduler._LRScheduler):
    """
    Cosine annealing with linear warmup.

    Warmup phase : LR increases linearly 0 -> base_lr over warmup_epochs.
    Cosine phase : LR decays base_lr -> lr_min following a cosine curve.

    Warmup prevents divergence at the start of training when the head
    weights are randomly initialised and gradients are large.
    """
    def __init__(self, optimiser: torch.optim.Optimizer, warmup_epochs: int,
                 total_epochs: int, lr_min: float = 1e-6, last_epoch: int = -1):
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.lr_min        = lr_min
        super().__init__(optimiser, last_epoch)

    def get_lr(self) -> List[float]:
        epoch = self.last_epoch
        lrs   = []
        for base_lr in self.base_lrs:
            if epoch < self.warmup_epochs:
                # Linear warmup
                scale = (epoch + 1) / max(self.warmup_epochs, 1)
            else:
                # Cosine decay
                progress = (epoch - self.warmup_epochs) / max(
                    self.total_epochs - self.warmup_epochs, 1
                )
                scale = (self.lr_min / max(base_lr, 1e-12) +
                         0.5 * (1 - self.lr_min / max(base_lr, 1e-12)) *
                         (1 + math.cos(math.pi * progress)))
            lrs.append(base_lr * scale)
        return lrs


def build_scheduler(optimiser: torch.optim.Optimizer, cfg: Config,
                    steps_per_epoch: int = 0):
    """Factory: instantiate and return the configured LR scheduler."""
    if cfg.SCHEDULER == "cosine_warmup":
        return CosineWarmupScheduler(
            optimiser, warmup_epochs=cfg.WARMUP_EPOCHS,
            total_epochs=cfg.EPOCHS, lr_min=cfg.LR_MIN
        )
    elif cfg.SCHEDULER == "onecycle":
        assert steps_per_epoch > 0, "steps_per_epoch must be provided for OneCycleLR"
        return torch.optim.lr_scheduler.OneCycleLR(
            optimiser,
            max_lr         = [g["lr"] for g in optimiser.param_groups],
            epochs         = cfg.EPOCHS,
            steps_per_epoch= steps_per_epoch,
            pct_start      = 0.1,
            anneal_strategy= "cos",
            div_factor     = 25.0,
            final_div_factor= 1e4,
        )
    else:
        raise ValueError(f"Unknown scheduler: {cfg.SCHEDULER}")


# ===========================================================================

<a id="section-18"></a>

## 🎲 Section 18 — Training Tricks (EMA, Mixup, CutMix)

### EMA — Exponential Moving Average
`θ_ema ← decay × θ_ema + (1 − decay) × θ_model`

EMA weights average out SGD noise → better generalisation at test time.
`decay = 0.9998` means the EMA lags ~5 000 update steps behind the live model.
**All validation and test inference uses the EMA model.**

### Mixup (Zhang et al., 2018)
Linearly interpolates two images *and* their labels:
`x_mix = λ·x_i + (1−λ)·x_j` — encourages linear interpolation in feature space.

### CutMix (Yun et al., 2019)
Pastes a rectangular crop from one image into another.
Stronger regulariser than Mixup because it removes content, forcing the model
to rely on multiple spatial regions (improves localisation).

### Label Smoothing
Applied inside every loss function: replaces hard targets with `{ε/2, 1−ε/2}`.

In [35]:
# SECTION 18A — EMA Model (Training Trick)
# ===========================================================================

class ModelEMA:
    """
    Maintains an Exponential Moving Average copy of model weights.

    EMA models generalise better than the final trained checkpoint because
    they average out SGD noise accumulated at the end of training.
    We use the EMA model for all validation and test inference.

    Update rule:
        theta_ema <- decay * theta_ema + (1 - decay) * theta_model
    """

    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay     = decay
        self.ema_model = copy.deepcopy(model)
        self.ema_model.eval()
        for p in self.ema_model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for ema_p, model_p in zip(self.ema_model.parameters(),
                                   model.parameters()):
            ema_p.copy_(self.decay * ema_p + (1 - self.decay) * model_p.data)

    def state_dict(self) -> dict:
        return self.ema_model.state_dict()


# ===========================================================================
# SECTION 18B — Mixup & CutMix (Training Tricks)
# ===========================================================================

def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.4) -> Tuple[torch.Tensor, torch.Tensor,
                                            torch.Tensor, float]:
    """
    Mixup augmentation (Zhang et al., 2018).
    Linearly interpolates two images and their labels.
    Encourages linear behaviour between training examples and acts as
    a powerful regulariser, especially for imbalanced datasets.
    """
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B     = x.size(0)
    idx   = torch.randperm(B, device=x.device)
    mixed = lam * x + (1 - lam) * x[idx]
    return mixed, y, y[idx], lam


def cutmix_data(x: torch.Tensor, y: torch.Tensor,
                alpha: float = 1.0) -> Tuple[torch.Tensor, torch.Tensor,
                                             torch.Tensor, float]:
    """
    CutMix augmentation (Yun et al., 2019).
    Pastes a rectangular region from one image into another.
    Stronger regulariser than Mixup because it removes content, forcing
    the model to rely on multiple regions (better localisation).
    """
    lam        = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    B, C, H, W = x.size()
    idx        = torch.randperm(B, device=x.device)
    cut_ratio  = math.sqrt(1 - lam)
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    cx, cy     = random.randint(0, W), random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)
    mixed      = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam        = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, y, y[idx], lam


def mixup_criterion(criterion: nn.Module, logits: torch.Tensor,
                    y_a: torch.Tensor, y_b: torch.Tensor,
                    lam: float) -> torch.Tensor:
    """Compute mixed loss: lambda * L(y_a) + (1-lambda) * L(y_b)."""
    return lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)


# ===========================================================================

<a id="section-11"></a>

## 🏋️ Section 11 — Training Loop

The `Trainer` class ties together every modern training technique:

```
for epoch in range(EPOCHS):
    for step, (images, labels) in enumerate(train_loader):
        apply Mixup or CutMix
        forward pass (AMP autocast)
        loss = ASL(logits, labels) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (step+1) % ACCUM_STEPS == 0:
            unscale → clip_grad_norm → scaler.step → scaler.update
            ema.update(model)
    scheduler.step()
    validate with EMA model
    save checkpoint if best AUC
    early stop if no improvement for PATIENCE epochs
```

### Key flags
| Technique | Config key | Default |
|-----------|-----------|---------|
| Automatic Mixed Precision | `CFG.AMP` | `True` |
| Gradient accumulation | `CFG.ACCUM_STEPS` | `2` |
| Gradient clipping | `CFG.GRAD_CLIP` | `1.0` |
| EMA decay | `CFG.EMA_DECAY` | `0.9998` |
| Early stopping patience | `CFG.PATIENCE` | `7` epochs |

In [48]:
# SECTION 11 — Training Loop
# ===========================================================================
from tqdm.auto import tqdm

class EarlyStopping:
    """Stops training if monitored metric does not improve for `patience` epochs."""

    def __init__(self, patience: int = 7, min_delta: float = 1e-4,
                 mode: str = "max"):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = -np.inf if mode == "max" else np.inf
        self.counter   = 0
        self.stop      = False

    def __call__(self, value: float) -> bool:
        improved = (value > self.best + self.min_delta) if self.mode == "max" \
                   else (value < self.best - self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


class Trainer:
    """
    Orchestrates the complete training loop including:
      - AMP (Automatic Mixed Precision) with GradScaler
      - Gradient accumulation (effective_batch = batch_size x accum_steps)
      - Gradient clipping (prevents exploding gradients in early training)
      - EMA (Exponential Moving Average) model tracking
      - Mixup / CutMix data augmentation
      - Early stopping on validation AUC
      - Best checkpoint saving
      - Optional WandB metric logging
    """

    def __init__(self, model: nn.Module, train_loader: DataLoader,
                 val_loader: DataLoader, criterion: nn.Module,
                 optimiser: torch.optim.Optimizer, scheduler,
                 cfg: Config, evaluator: "Evaluator"):
        self.model        = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.criterion    = criterion.to(DEVICE)
        self.optimiser    = optimiser
        self.scheduler    = scheduler
        self.cfg          = cfg
        self.evaluator    = evaluator

        self.ema          = ModelEMA(model, cfg.EMA_DECAY)
        self.scaler       = GradScaler(enabled=cfg.AMP)
        self.early_stop   = EarlyStopping(cfg.PATIENCE, cfg.MIN_DELTA, mode="max")

        self.history: Dict[str, List] = defaultdict(list)
        self.best_auc  = 0.0
        self.best_epoch = 0
        self.start_time = None

        # ----------------------------------------------------------
        # Load pretrained / resume checkpoint
        # Priority:
        # 1- Local best_model.pth
        # 2- Kaggle pretrained model
        # ----------------------------------------------------------
        local_ckpt = self.cfg.CHECKPOINT_DIR / "best_model.pth"
        kaggle_ckpt = Path(
            "/kaggle/input/notebooks/agnefits/clinicai-v2-0/checkpoints/best_model.pth"
        )

        ckpt_path = None

        if local_ckpt.exists():
            ckpt_path = local_ckpt
            print(f"Resuming from local checkpoint: {local_ckpt}")

        elif kaggle_ckpt.exists():
            ckpt_path = kaggle_ckpt
            print(f"Loading pretrained Kaggle model: {kaggle_ckpt}")

        if ckpt_path is not None:
            ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

            if "ema" in ckpt:
                self.model.load_state_dict(ckpt["ema"])
                self.ema.ema_model.load_state_dict(ckpt["ema"])
            else:
                self.model.load_state_dict(ckpt["model"])
                self.ema.ema_model.load_state_dict(ckpt["model"])

            print(f"Loaded checkpoint (epoch {ckpt.get('epoch', '?')})")

        else:
            print("No checkpoint found. Training from scratch.")

        # Memory format optimisation
        if cfg.CHANNELS_LAST and DEVICE.type == "cuda":
            self.model = self.model.to(memory_format=torch.channels_last)

        # torch.compile (PyTorch >= 2.0)
        if cfg.COMPILE and hasattr(torch, "compile") and DEVICE.type == "cuda":
            print("Applying torch.compile()...")
            self.model = torch.compile(self.model)

        # WandB initialisation
        if cfg.WANDB and WANDB_AVAILABLE:
            wandb.init(project=cfg.WANDB_PROJECT, name=cfg.WANDB_RUN_NAME,
                       config=vars(cfg))

    # ------------------------------------------------------------------
    def _train_one_epoch(self, epoch: int) -> Dict[str, float]:
        """Run one full pass through the training data."""
        self.model.train()
        total_loss = 0.0
        n_batches  = 0
        self.optimiser.zero_grad()

        epoch_start = time.time()

        pbar = tqdm(
            enumerate(self.train_loader),
            total=len(self.train_loader),
            desc=f"Epoch {epoch}/{self.cfg.EPOCHS}",
            leave=False,
        )

        for step, (images, labels) in pbar:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if self.cfg.CHANNELS_LAST and DEVICE.type == "cuda":
                images = images.to(memory_format=torch.channels_last)

            # ── Mixup / CutMix (50/50 when both are enabled) ─────────
            use_mix = False
            if self.cfg.USE_MIXUP and self.cfg.USE_CUTMIX:
                use_mix    = True
                use_cutmix = random.random() < 0.5
            elif self.cfg.USE_MIXUP:
                use_mix, use_cutmix = True, False
            elif self.cfg.USE_CUTMIX:
                use_mix, use_cutmix = True, True

            if use_mix:
                if use_cutmix:
                    images, y_a, y_b, lam = cutmix_data(images, labels,
                                                         self.cfg.CUTMIX_ALPHA)
                else:
                    images, y_a, y_b, lam = mixup_data(images, labels,
                                                        self.cfg.MIXUP_ALPHA)

            # ── Forward pass + AMP ───────────────────────────────────
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(images)
                if use_mix:
                    loss = mixup_criterion(self.criterion, logits, y_a, y_b, lam)
                else:
                    loss = self.criterion(logits, labels)
                loss = loss / self.cfg.ACCUM_STEPS      # scale for accumulation

            # ── Backward pass ────────────────────────────────────────
            self.scaler.scale(loss).backward()

            # ── Gradient accumulation: update every ACCUM_STEPS ─────
            if (step + 1) % self.cfg.ACCUM_STEPS == 0 or \
               (step + 1) == len(self.train_loader):
                # Must unscale before clipping for AMP compatibility
                self.scaler.unscale_(self.optimiser)
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.cfg.GRAD_CLIP
                )
                self.scaler.step(self.optimiser)
                self.scaler.update()
                self.optimiser.zero_grad()
                self.ema.update(self.model)     # update EMA after each step

            epoch_time = time.time() - epoch_start

            pbar.set_postfix({
                "loss": f"{total_loss / max(n_batches,1):.4f}",
                "lr": f"{self.optimiser.param_groups[-1]['lr']:.2e}"
            })
            
            total_loss += loss.item() * self.cfg.ACCUM_STEPS
            n_batches  += 1

        epoch_time = time.time() - epoch_start

        return {
            "loss": total_loss / max(n_batches, 1),
            "time": epoch_time,
        }

    # ------------------------------------------------------------------
    @torch.no_grad()
    def _validate(self) -> Dict[str, float]:
        """Evaluate EMA model on the validation set."""
        self.ema.ema_model.eval()
        all_logits, all_labels = [], []

        for images, labels in self.val_loader:
            images = images.to(DEVICE, non_blocking=True)
            if self.cfg.CHANNELS_LAST and DEVICE.type == "cuda":
                images = images.to(memory_format=torch.channels_last)
            with autocast(enabled=self.cfg.AMP):
                logits = self.ema.ema_model(images)
            all_logits.append(logits.cpu().float())
            all_labels.append(labels.cpu().float())

        all_logits = torch.cat(all_logits, dim=0)
        all_labels = torch.cat(all_labels, dim=0)
        probs      = torch.sigmoid(all_logits).numpy()
        labels_np  = all_labels.numpy()
        return self.evaluator.compute(labels_np, probs)

    # ------------------------------------------------------------------
    def _save_checkpoint(self, epoch: int, metrics: Dict,
                          is_best: bool) -> None:
        state = {
            "epoch"     : epoch,
            "model"     : self.model.state_dict(),
            "ema"       : self.ema.state_dict(),
            "optimiser" : self.optimiser.state_dict(),
            "scaler"    : self.scaler.state_dict(),
            "metrics"   : metrics,
            "cfg"       : vars(self.cfg),
        }
        if is_best:
            torch.save(state, self.cfg.CHECKPOINT_DIR / "best_model.pth")
            print(f"   Best model saved  (AUC={metrics['auc_macro']:.4f})")
    # ------------------------------------------------------------------
    def _log(self, epoch: int, train_m: Dict, val_m: Dict) -> None:
        lr = self.optimiser.param_groups[-1]["lr"]
        self.history["epoch"].append(epoch)
        self.history["train_loss"].append(train_m["loss"])
        self.history["val_auc"].append(val_m["auc_macro"])
        self.history["lr"].append(lr)

        print(
            f"Epoch [{epoch:3d}/{self.cfg.EPOCHS}] | "
            f"Loss: {train_m['loss']:.4f} | "
            f"AUC: {val_m['auc_macro']:.4f} | "
            f"AP: {val_m['map']:.4f} | "
            f"LR: {lr:.2e} | "
            f"Time: {train_m['time']:.1f}s"
        )

        if self.cfg.WANDB and WANDB_AVAILABLE:
            wandb.log({"epoch": epoch, **train_m,
                       **{f"val_{k}": v for k, v in val_m.items()},
                       "lr": lr})

    # ------------------------------------------------------------------
    def fit(self) -> Dict:
        """Main entry-point: run the full training loop, return history."""
        print(f"\n{'='*60}")
        print(f"  ClinicAI v2.1  |  {self.cfg.BACKBONE}  |  {self.cfg.LOSS.upper()}")
        print(f"  Epochs  : {self.cfg.EPOCHS}  |  "
              f"Batch : {self.cfg.BATCH_SIZE}x{self.cfg.ACCUM_STEPS}")
        print(f"  Device  : {DEVICE}")
        print(f"{'='*60}\n")

        self.start_time = time.time()

        for epoch in range(1, self.cfg.EPOCHS + 1):
            train_m = self._train_one_epoch(epoch)
            val_m   = self._validate()

            if isinstance(self.scheduler, CosineWarmupScheduler):
                self.scheduler.step()

            is_best = val_m["auc_macro"] > self.best_auc
            if is_best:
                self.best_auc   = val_m["auc_macro"]
                self.best_epoch = epoch

            self._save_checkpoint(epoch, val_m, is_best)
            self._log(epoch, train_m, val_m)

            if self.early_stop(val_m["auc_macro"]):
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(best AUC={self.best_auc:.4f} @ epoch {self.best_epoch})")
                break

        elapsed = time.time() - self.start_time
        print(f"\nTraining complete in {elapsed/60:.1f} min")
        if self.cfg.WANDB and WANDB_AVAILABLE:
            wandb.finish()
        return dict(self.history)


# ===========================================================================

<a id="section-12"></a>

## 📐 Section 12 — Metrics

`Evaluator` computes a full suite of multi-label metrics after every epoch
and produces three plots at the end of training.

| Metric | Notes |
|--------|-------|
| **ROC-AUC macro** | Primary metric — unaffected by threshold choice |
| **ROC-AUC micro** | Treats every (sample, label) pair independently |
| **mAP** | Area under the Precision-Recall curve (macro) |
| **Precision / Recall / F1** | Macro-averaged, threshold-dependent |
| **Per-class AUC** | Allows identifying the weakest classes |

### Saved Plots
- `per_class_auc.png` — sorted bar chart coloured by AUC value
- `roc_curves.png` — individual ROC curves for all 14 classes
- `confusion_matrices.png` — TP/FP/TN/FN for each class at threshold 0.5

In [37]:
# SECTION 12 — Metrics
# ===========================================================================

class Evaluator:
    """
    Computes a comprehensive set of classification metrics for multi-label
    chest X-ray classification.

    Metrics produced
    ----------------
    * ROC-AUC macro   — primary ranking metric (averaged over classes)
    * ROC-AUC micro   — all labels treated as one binary problem
    * mAP             — mean Average Precision
    * Precision, Recall, F1 (macro)
    * Per-class ROC-AUC
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg

    # ------------------------------------------------------------------
    def compute(self, y_true: np.ndarray, y_prob: np.ndarray,
                thresholds: Optional[np.ndarray] = None) -> Dict:
        """
        y_true     : [N, 14]  ground-truth binary labels
        y_prob     : [N, 14]  predicted probabilities in [0, 1]
        thresholds : [14]     per-class thresholds (default 0.5)
        """
        if thresholds is None:
            thresholds = np.full(self.cfg.NUM_CLASSES, 0.5)
        y_pred = (y_prob >= thresholds).astype(int)

        metrics = {}

        # ── ROC-AUC ─────────────────────────────────────────────────────
        try:
            metrics["auc_macro"] = roc_auc_score(y_true, y_prob, average="macro")
        except ValueError:
            metrics["auc_macro"] = 0.0

        try:
            metrics["auc_micro"] = roc_auc_score(
                y_true.ravel(), y_prob.ravel(), average=None
            )
        except ValueError:
            metrics["auc_micro"] = 0.0

        # Per-class AUC (skip classes with no positive samples)
        per_class_auc = {}
        for i, cls in enumerate(self.cfg.CLASSES):
            n_pos = y_true[:, i].sum()
            if 0 < n_pos < len(y_true):
                try:
                    per_class_auc[cls] = roc_auc_score(y_true[:, i], y_prob[:, i])
                except ValueError:
                    per_class_auc[cls] = 0.0
            else:
                per_class_auc[cls] = 0.0
        metrics["per_class_auc"] = per_class_auc

        # ── Average Precision ────────────────────────────────────────────
        try:
            metrics["map"] = average_precision_score(y_true, y_prob, average="macro")
        except ValueError:
            metrics["map"] = 0.0

        # ── Precision / Recall / F1 ──────────────────────────────────────
        metrics["precision"] = precision_score(y_true, y_pred, average="macro",
                                               zero_division=0)
        metrics["recall"]    = recall_score(y_true, y_pred, average="macro",
                                            zero_division=0)
        metrics["f1"]        = f1_score(y_true, y_pred, average="macro",
                                        zero_division=0)
        return metrics

    # ------------------------------------------------------------------
    def plot_per_class_auc(self, per_class_auc: Dict[str, float],
                           save_path: Optional[Path] = None) -> None:
        classes = list(per_class_auc.keys())
        aucs    = list(per_class_auc.values())
        idx     = np.argsort(aucs)[::-1]

        fig, ax = plt.subplots(figsize=(12, 5))
        colors  = plt.cm.RdYlGn(np.array(aucs)[idx] - 0.5)
        bars    = ax.bar(np.array(classes)[idx], np.array(aucs)[idx], color=colors)
        ax.axhline(np.mean(aucs), color="navy", linestyle="--", lw=1.5,
                   label=f"Macro AUC = {np.mean(aucs):.4f}")
        ax.set_ylim([0.5, 1.0])
        ax.set_ylabel("ROC-AUC")
        ax.set_title("Per-class ROC-AUC", fontweight="bold")
        ax.set_xticklabels(np.array(classes)[idx], rotation=45, ha="right")
        for bar, val in zip(bars, np.array(aucs)[idx]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=8)
        ax.legend()
        plt.tight_layout()
        path = save_path or CFG.OUTPUT_DIR / "per_class_auc.png"
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved: {path.name}")

    # ------------------------------------------------------------------
    def plot_roc_curves(self, y_true: np.ndarray, y_prob: np.ndarray,
                        save_path: Optional[Path] = None) -> None:
        fig, axes = plt.subplots(2, 7, figsize=(26, 8))
        axes      = axes.flatten()
        for i, cls in enumerate(self.cfg.CLASSES):
            if y_true[:, i].sum() == 0:
                axes[i].axis("off")
                continue
            fpr, tpr, _ = roc_curve(y_true[:, i], y_prob[:, i])
            auc_val     = roc_auc_score(y_true[:, i], y_prob[:, i])
            axes[i].plot(fpr, tpr, color="#E74C3C", lw=2,
                         label=f"AUC={auc_val:.3f}")
            axes[i].plot([0, 1], [0, 1], "k--", lw=1)
            axes[i].set_title(cls, fontsize=9, fontweight="bold")
            axes[i].set_xlabel("FPR")
            axes[i].set_ylabel("TPR")
            axes[i].legend(fontsize=8)
            axes[i].set_xlim([0, 1])
            axes[i].set_ylim([0, 1])
        plt.suptitle("Per-class ROC Curves", fontsize=14, fontweight="bold")
        plt.tight_layout()
        path = save_path or CFG.OUTPUT_DIR / "roc_curves.png"
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved: {path.name}")

    # ------------------------------------------------------------------
    def plot_confusion_matrices(self, y_true: np.ndarray, y_pred: np.ndarray,
                                 save_path: Optional[Path] = None) -> None:
        fig, axes = plt.subplots(2, 7, figsize=(26, 8))
        axes      = axes.flatten()
        for i, cls in enumerate(self.cfg.CLASSES):
            cm = confusion_matrix(y_true[:, i], y_pred[:, i])
            if cm.shape != (2, 2):
                axes[i].axis("off")
                continue
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[i],
                        cbar=False, xticklabels=["Neg", "Pos"],
                        yticklabels=["Neg", "Pos"])
            axes[i].set_title(cls, fontsize=8, fontweight="bold")
        plt.suptitle("Confusion Matrices (per class)", fontsize=14, fontweight="bold")
        plt.tight_layout()
        path = save_path or CFG.OUTPUT_DIR / "confusion_matrices.png"
        plt.savefig(path, dpi=150); plt.close()
        print(f"Saved: {path.name}")


# ===========================================================================

<a id="section-13"></a>

## 🎯 Section 13 — Per-class Threshold Optimisation

Using a **fixed threshold of 0.5** is suboptimal for imbalanced multi-label data:
- Rare classes (Hernia, Pneumonia) are systematically under-predicted.
- Each class has a different optimal operating point on its ROC curve.

### Youden's J Statistic
For each class independently, we search the threshold that maximises:

$$J = \text{TPR} - \text{FPR} = \text{Sensitivity} + \text{Specificity} - 1$$

This simultaneously maximises both sensitivity and specificity.

### Result
- Optimal thresholds are saved to `thresholds.json`.
- F1 typically improves by **5–15%** compared to fixed 0.5.
- AUC is unaffected (threshold-independent), but precision/recall improve.

In [38]:
# SECTION 13 — Threshold Optimisation
# ===========================================================================

class ThresholdOptimiser:
    """
    Finds the optimal decision threshold for each class independently
    using Youden's J statistic on the validation set:

        J = Sensitivity + Specificity - 1 = TPR - FPR

    This maximises the geometric performance of both TPR and TNR,
    which is appropriate for imbalanced multi-label medical data.
    Using a fixed 0.5 threshold is suboptimal because:
    1. Rare diseases are systematically under-predicted (low recall).
    2. Different classes have different optimal operating points.
    """

    def __init__(self, cfg: Config):
        self.cfg        = cfg
        self.thresholds = np.full(cfg.NUM_CLASSES, 0.5)

    def fit(self, y_true: np.ndarray, y_prob: np.ndarray) -> np.ndarray:
        """Compute optimal per-class thresholds from validation predictions."""
        for i, cls in enumerate(self.cfg.CLASSES):
            n_pos = y_true[:, i].sum()
            if n_pos == 0 or n_pos == len(y_true):
                self.thresholds[i] = 0.5
                continue
            fpr, tpr, candidates = roc_curve(y_true[:, i], y_prob[:, i])
            j_stat               = tpr - fpr
            best_j               = np.argmax(j_stat)
            self.thresholds[i]   = candidates[best_j]

        print("\nOptimal Thresholds (Youden's J):")
        for cls, t in zip(self.cfg.CLASSES, self.thresholds):
            print(f"  {cls:<22} : {t:.4f}")
        return self.thresholds

    def save(self, path: Optional[Path] = None) -> None:
        path = path or CFG.OUTPUT_DIR / "thresholds.json"
        with open(path, "w") as f:
            json.dump(dict(zip(self.cfg.CLASSES, self.thresholds.tolist())), f, indent=2)
        print(f"Thresholds saved to {path}")

    def load(self, path: Path) -> np.ndarray:
        with open(path) as f:
            d = json.load(f)
        self.thresholds = np.array([d[cls] for cls in self.cfg.CLASSES])
        return self.thresholds


# ===========================================================================

<a id="section-14"></a>

## 🔄 Section 14 — Test-Time Augmentation (TTA)

TTA averages predictions from multiple augmented views of the same image,
reducing prediction variance without any retraining.

### Augmentation Views
For each scale in `CFG.TTA_SCALES = [288, 320, 352]`:
1. **Original** image at that scale
2. **Horizontally flipped** image at that scale

→ **6 total forward passes** per image.

### Expected Gain
TTA typically boosts macro AUC by **0.3–0.8 percentage points**.

> **Batch mode** (for submission) uses only the standard scale for speed.

In [39]:
# SECTION 14 — Test-Time Augmentation (TTA)
# ===========================================================================

class TTAInference:
    """
    Test-Time Augmentation averages predictions from multiple augmented
    views of the same image to reduce prediction variance.

    Augmentation views
    ------------------
    For each scale in CFG.TTA_SCALES:
      1. Original image at that scale
      2. Horizontally flipped image at that scale
    Total forward passes: 2 x len(TTA_SCALES) per image.

    Averaging predictions typically boosts macro AUC by 0.3-0.8 pp
    without any additional training.
    """

    def __init__(self, model: nn.Module, cfg: Config):
        self.model = model.eval()
        self.cfg   = cfg

    @torch.no_grad()
    def predict(self, image_path: Union[str, Path]) -> np.ndarray:
        """Predict probabilities for a single image with multi-scale TTA."""
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {image_path}")
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        all_probs = []
        for scale in self.cfg.TTA_SCALES:
            tfm = build_tta_transforms(scale, self.cfg)

            # Original
            t_img  = tfm(image=img)["image"].unsqueeze(0).to(DEVICE)
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(t_img)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())

            # Horizontal flip
            img_f  = cv2.flip(img, 1)
            t_imgf = tfm(image=img_f)["image"].unsqueeze(0).to(DEVICE)
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(t_imgf)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())

        return np.mean(all_probs, axis=0).squeeze(0)    # [NUM_CLASSES]

    @torch.no_grad()
    def predict_loader(self, loader: DataLoader) -> np.ndarray:
        """Batch inference (standard resolution, no multi-scale for speed)."""
        all_probs = []
        self.model.eval()
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(images)
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(all_probs, axis=0)


# ===========================================================================

<a id="section-15"></a>

## 🔥 Section 15 — Explainability (GradCAM)

**GradCAM** (Selvaraju et al., 2017) computes the gradient of the class score
with respect to the last convolutional feature map, producing a heatmap that
highlights *which spatial regions* drove the prediction.

### Clinical Value
In chest X-rays, GradCAM typically highlights:
- Lung apices for Pneumothorax
- Cardiac silhouette for Cardiomegaly
- Costophrenic angles for Effusion

This provides **radiologist-level interpretability** and helps identify
when the model is focusing on spurious correlations.

### Saved output
`/kaggle/working/gradcam/gradcam_<image_name>.png` — one panel per disease class.

In [40]:
# SECTION 15 — Explainability (GradCAM)
# ===========================================================================

class GradCAMVisualiser:
    """
    Generates highly focused Heatmaps using GradCAM++ with Percentile Clipping.
    
    Upgrades from standard GradCAM:
    1. Uses GradCAM++ for better localization of multiple lesions and fine details.
    2. Applies Percentile Clipping to eliminate low-activation background noise.
    3. Uses Gaussian Smoothing to create a focused, clinically readable hotspot.
    """

    def __init__(self, model: nn.Module, cfg: Config):
        self.model = model
        self.cfg   = cfg
        self._cam  = None

        if not GRADCAM_AVAILABLE:
            print("GradCAM library not available.")
            return

        target_layer = model.get_gradcam_layer()
        # استخدام GradCAMPlusPlus بدلاً من GradCAM العادية لتركيز أقوى
        self._cam    = GradCAMPlusPlus(model=model, target_layers=[target_layer])

    def visualise(self, image_path: Union[str, Path],
                  class_indices: Optional[List[int]] = None,
                  save_dir: Optional[Path] = None) -> None:
        
        if not GRADCAM_AVAILABLE or self._cam is None:
            return

        save_dir = save_dir or self.cfg.GRAD_CAM_DIR
        save_dir.mkdir(parents=True, exist_ok=True)

        if class_indices is None:
            class_indices = list(range(self.cfg.NUM_CLASSES))

        img_bgr  = cv2.imread(str(image_path))
        if img_bgr is None:
            print(f"Cannot read: {image_path}")
            return
        
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_resz = cv2.resize(img_rgb, (self.cfg.IMG_SIZE, self.cfg.IMG_SIZE))
        img_norm = img_resz.astype(np.float32) / 255.0

        tfm    = build_transforms("val", self.cfg)
        tensor = tfm(image=img_rgb)["image"]
        tensor = tensor.unsqueeze(0).to(DEVICE)

        self.model.eval()
        with torch.no_grad():
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(tensor)
        probs = torch.sigmoid(logits).cpu().numpy().squeeze()

        fig_cols = min(4, len(class_indices))
        fig_rows = math.ceil(len(class_indices) / fig_cols)
        fig, axes = plt.subplots(fig_rows, fig_cols,
                                 figsize=(5 * fig_cols, 5 * fig_rows))
        
        # التأكد من أن axes مصفوفة حتى لو كانت صورة واحدة
        if isinstance(axes, plt.Axes):
            axes = np.array([axes])
        axes = axes.flatten()

        for plot_idx, cls_idx in enumerate(class_indices):
            targets = [ClassifierOutputTarget(cls_idx)]
            cam_map = self._cam(input_tensor=tensor, targets=targets)[0]

            # -------------------------------------------------------------
            # التعديلات الطبية المتقدمة (Medical Imaging Enhancements):
            # 1. Percentile Clipping: إزالة التشويش والإشارات الضعيفة (الخلفية)
            # نأخذ فقط أعلى 25% من التنشيطات (Signals) ونصفر الباقي
            p_val = np.percentile(cam_map, 75) 
            cam_map = np.where(cam_map < p_val, 0, cam_map)

            # 2. Gaussian Smoothing: تنعيم الحواف لتبدو كبؤرة مرضية طبيعية
            cam_map = cv2.GaussianBlur(cam_map, (15, 15), 0)

            # 3. Re-normalization: إعادة ضبط القيم لتصبح بين 0 و 1 بعد القص
            if np.max(cam_map) != 0:
                cam_map = cam_map / np.max(cam_map)
            # -------------------------------------------------------------

            overlay = show_cam_on_image(img_norm, cam_map, use_rgb=True)
            axes[plot_idx].imshow(overlay)
            axes[plot_idx].set_title(
                f"{self.cfg.CLASSES[cls_idx]}\n(p={probs[cls_idx]:.3f})",
                fontsize=9, fontweight="bold"
            )
            axes[plot_idx].axis("off")

        for ax in axes[len(class_indices):]:
            ax.axis("off")

        img_name  = Path(image_path).stem
        plt.suptitle(f"GradCAM++ (Enhanced) - {img_name}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        save_path = save_dir / f"gradcam_enhanced_{img_name}.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"Enhanced GradCAM saved: {save_path}")


# ===========================================================================

<a id="section-16"></a>

## 🚀 Section 16 — Inference Pipeline

`InferencePipeline` is the production-ready interface for deployment.

### Methods

| Method | Input | Output |
|--------|-------|--------|
| `predict_single(path, use_tta=True)` | One image path | Dict: probs + preds + disease names |
| `predict_batch(df, image_dir)` | DataFrame | `[N, 14]` probability matrix |
| `generate_submission(df, image_dir)` | DataFrame | Kaggle CSV saved to disk |

### Submission Format
```
Image Index, Atelectasis, Cardiomegaly, ..., Hernia
00000001_000.png, 0, 1, ..., 0
```

In [41]:
# SECTION 16 — Inference Pipeline
# ===========================================================================

class InferencePipeline:
    """
    Production-grade inference module supporting:
      - Single-image prediction with optional TTA
      - Batch prediction from a DataFrame
      - Kaggle submission CSV generation
    """

    def __init__(self, model: nn.Module, cfg: Config,
                 thresholds: Optional[np.ndarray] = None):
        self.model      = model.eval()
        self.cfg        = cfg
        self.thresholds = (thresholds if thresholds is not None
                           else np.full(cfg.NUM_CLASSES, 0.5))
        self.tta        = TTAInference(model, cfg)
        self.transform  = build_transforms("val", cfg)

    def predict_single(self, image_path: Union[str, Path],
                       use_tta: bool = True) -> Dict:
        """Single-image prediction with optional TTA."""
        if use_tta:
            probs = self.tta.predict(image_path)
        else:
            img    = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
            img    = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            tensor = self.transform(image=img)["image"].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                with autocast(enabled=self.cfg.AMP):
                    logits = self.model(tensor)
            probs = torch.sigmoid(logits).cpu().numpy().squeeze()

        predictions = (probs >= self.thresholds).astype(int)
        detected    = [cls for cls, pred in zip(self.cfg.CLASSES, predictions)
                       if pred]
        return {
            "probabilities"  : dict(zip(self.cfg.CLASSES, probs.tolist())),
            "predictions"    : dict(zip(self.cfg.CLASSES, predictions.tolist())),
            "detected_labels": detected if detected else ["No Finding"],
        }

    @torch.no_grad()
    def predict_batch(self, df: pd.DataFrame, image_dir: Path) -> np.ndarray:
        """Batch inference; returns [N, NUM_CLASSES] probability matrix."""
        dataset = ChestXray14Dataset(df, image_dir, self.transform, self.cfg)
        loader  = DataLoader(dataset, batch_size=self.cfg.BATCH_SIZE,
                             shuffle=False, num_workers=self.cfg.NUM_WORKERS,
                             pin_memory=self.cfg.PIN_MEMORY)
        probs   = []
        self.model.eval()
        for images, _ in loader:
            images = images.to(DEVICE, non_blocking=True)
            with autocast(enabled=self.cfg.AMP):
                logits = self.model(images)
            probs.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(probs, axis=0)

    def generate_submission(self, df: pd.DataFrame, image_dir: Path,
                             save_path: Optional[Path] = None) -> pd.DataFrame:
        """Generate a Kaggle-style submission CSV."""
        print(f"Generating submission for {len(df):,} images...")
        probs   = self.predict_batch(df, image_dir)
        preds   = (probs >= self.thresholds).astype(int)
        result  = pd.DataFrame(preds, columns=self.cfg.CLASSES)
        result.insert(0, "Image Index", df["Image Index"].values)
        save_path = save_path or self.cfg.OUTPUT_DIR / "submission.csv"
        result.to_csv(save_path, index=False)
        print(f"Submission saved to {save_path}  ({len(result):,} rows)")
        return result


# ===========================================================================

<a id="section-17"></a>

## 🔍 Section 17 — Error Analysis

`ErrorAnalyser` systematically identifies *how* the model fails for each class.

| Category | Definition | Clinical Meaning |
|----------|-----------|-----------------|
| **False Positives** | Predicted positive, GT negative | Over-diagnosis — unnecessary follow-up |
| **False Negatives** | Predicted negative, GT positive | Under-diagnosis — missed disease ⚠️ |
| **Hard Examples** | Highest binary cross-entropy | Cases where model is most wrong |
| **Most Uncertain** | Predicted prob closest to threshold | Cases where model is least confident |

Each category produces a saved figure grid.  
Use these to guide **data collection**, **augmentation tuning**, and **threshold adjustment**.

In [42]:
# SECTION 17 — Error Analysis
# ===========================================================================

class ErrorAnalyser:
    """
    Analyses model failures to gain actionable diagnostic insights.

    Categories
    ----------
    * False Positives : model over-diagnosed a disease that is absent
    * False Negatives : model missed a disease that is present
    * Hard Examples   : highest binary-cross-entropy loss (hardest for model)
    * Most Uncertain  : predicted probability closest to the threshold
    """

    def __init__(self, df: pd.DataFrame, cfg: Config):
        self.df  = df
        self.cfg = cfg

    def analyse(self, y_true: np.ndarray, y_prob: np.ndarray,
                thresholds: np.ndarray, image_dir: Path, n: int = 4) -> None:
        """Run error analysis and save visualisations for each class."""
        y_pred = (y_prob >= thresholds).astype(int)

        for cls_idx, cls in enumerate(self.cfg.CLASSES):
            gt   = y_true[:, cls_idx]
            prob = y_prob[:, cls_idx]
            pred = y_pred[:, cls_idx]

            fp_idx    = np.where((pred == 1) & (gt == 0))[0]
            fn_idx    = np.where((pred == 0) & (gt == 1))[0]
            bce_loss  = -(gt * np.log(prob + 1e-8) + (1-gt) * np.log(1-prob+1e-8))
            hard_idx  = np.argsort(bce_loss)[::-1][:n]
            margin    = np.abs(prob - thresholds[cls_idx])
            uncert_idx = np.argsort(margin)[:n]

            print(f"\n{'-'*50}")
            print(f"  {cls:<22}  |  FP={len(fp_idx)}  FN={len(fn_idx)}")
            print(f"{'-'*50}")

            for label, indices in [
                ("False_Positives", fp_idx[:n]),
                ("False_Negatives", fn_idx[:n]),
                ("Hard_Examples",   hard_idx),
                ("Most_Uncertain",  uncert_idx),
            ]:
                if len(indices) == 0:
                    continue
                self._show_examples(label, indices, cls, prob, gt, image_dir)

    def _show_examples(self, label: str, indices: np.ndarray, cls: str,
                        prob: np.ndarray, gt: np.ndarray,
                        image_dir: Path) -> None:
        fig, axes = plt.subplots(1, len(indices),
                                 figsize=(4 * len(indices), 4))
        if len(indices) == 1:
            axes = [axes]
        for ax, idx in zip(axes, indices):
            img_name = self.df.iloc[idx]["Image Index"]
            img_path = image_dir / img_name
            if img_path.exists():
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                ax.imshow(img, cmap="gray")
            ax.set_title(f"GT={int(gt[idx])}  P={prob[idx]:.3f}", fontsize=9)
            ax.axis("off")
        plt.suptitle(f"{label.replace('_', ' ')} — {cls}", fontweight="bold")
        plt.tight_layout()
        fname = f"error_{label.lower()}_{cls.lower()}.png"
        plt.savefig(CFG.OUTPUT_DIR / fname, dpi=120)
        plt.close()


# ===========================================================================

<a id="section-19"></a>

## ⚡ Section 19 — Performance Optimisation

Hardware-level optimisations for Kaggle P100 / T4 / L4 GPUs.

### Channels-Last Memory Format
NVIDIA convolution kernels run faster on NHWC (channels last) vs NCHW layout.
`model.to(memory_format=torch.channels_last)` enables this transparently.

### `torch.compile` (PyTorch ≥ 2.0)
Traces the model graph, fuses operators, and generates optimised Triton/CUDA kernels.
10–40% speedup at the cost of ~2 min one-time compilation.
**Disabled by default** (`CFG.COMPILE = False`) — enable for long training runs.

### `cudnn.benchmark = True`
Auto-selects the fastest cuDNN convolution algorithm for each input shape.
Large speedup for fixed image sizes (our case: always 320×320).

### `non_blocking=True`
All `.to(DEVICE, non_blocking=True)` calls in the DataLoader allow the GPU to
overlap data transfer with computation.

In [43]:
# SECTION 19 — Performance Optimisation
# ===========================================================================

def apply_performance_opts(model: nn.Module, cfg: Config) -> nn.Module:
    """
    Apply hardware-level optimisations for Kaggle P100/T4/L4 GPUs.

    Channels-last memory format
    ----------------------------
    NVIDIA convolution kernels run faster with NHWC layout (channels last)
    vs NCHW (channels first). torch.channels_last enables this transparently.

    torch.compile
    -------------
    PyTorch 2.0+ traces the model graph and generates optimised Triton/CUDA
    kernels with operator fusion. 10-40% speedup at the cost of ~2min warmup.

    cudnn.benchmark
    ---------------
    Automatically selects the fastest cuDNN conv algorithm for each input
    shape at the start of training. Recommended for fixed image sizes.
    """
    if cfg.CHANNELS_LAST and DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)
        print("channels_last memory format enabled")

    if cfg.COMPILE and hasattr(torch, "compile") and DEVICE.type == "cuda":
        model = torch.compile(model, mode="reduce-overhead")
        print("torch.compile() applied")

    torch.backends.cudnn.benchmark = cfg.CUDNN_BENCHMARK
    if cfg.CUDNN_BENCHMARK:
        print("cudnn.benchmark = True")

    return model


# ===========================================================================

<a id="section-20"></a>

## 📋 Section 20 — Final Report

`Reporter` prints a comprehensive summary after training completes.

### Output
- Best epoch & total training time
- Peak GPU memory usage
- Total & trainable parameter count
- Macro / Micro ROC-AUC
- Mean Average Precision
- Per-class AUC table with optimal thresholds
- `training_curves.png` — loss and AUC curves over epochs

In [44]:
# SECTION 20 — Final Report
# ===========================================================================

class Reporter:
    """Generates the comprehensive final performance report after training."""

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def print_report(self, history: Dict, best_metrics: Dict,
                     thresholds: np.ndarray, model: nn.Module,
                     training_time_sec: float) -> None:
        print("\n" + "=" * 70)
        print("  CLINICAI v2.1 — FINAL REPORT")
        print("=" * 70)

        best_epoch = (history.get("epoch", [0])
                      [np.argmax(history.get("val_auc", [0]))])
        print(f"\n  Best Epoch        : {best_epoch}")
        print(f"  Training Time     : {training_time_sec/60:.1f} minutes")

        if DEVICE.type == "cuda":
            mem = torch.cuda.max_memory_allocated() / 1e9
            print(f"  Peak GPU Memory   : {mem:.2f} GB")
            torch.cuda.reset_peak_memory_stats()

        total_p     = model.param_count() / 1e6
        trainable_p = model.trainable_param_count() / 1e6
        print(f"  Total Params      : {total_p:.1f} M")
        print(f"  Trainable Params  : {trainable_p:.1f} M")

        print(f"\n  Macro ROC-AUC     : {best_metrics.get('auc_macro', 0):.4f}")
        print(f"  Micro ROC-AUC     : {best_metrics.get('auc_micro', 0):.4f}")
        print(f"  Mean Avg Prec     : {best_metrics.get('map', 0):.4f}")
        print(f"  Precision (macro) : {best_metrics.get('precision', 0):.4f}")
        print(f"  Recall    (macro) : {best_metrics.get('recall', 0):.4f}")
        print(f"  F1        (macro) : {best_metrics.get('f1', 0):.4f}")

        print("\n  Per-class ROC-AUC & Optimal Threshold:")
        print(f"  {'Class':<24} {'AUC':>7}   {'Threshold':>9}")
        print("  " + "-" * 46)
        per_cls = best_metrics.get("per_class_auc", {})
        for i, cls in enumerate(self.cfg.CLASSES):
            auc_val = per_cls.get(cls, 0.0)
            t_val   = thresholds[i]
            flag    = "STAR" if auc_val >= 0.85 else ("LOW" if auc_val < 0.75 else "   ")
            print(f"  {cls:<24} {auc_val:>7.4f}   {t_val:>9.4f}  {flag}")

        print(f"\n  Backbone          : {self.cfg.BACKBONE}")
        print(f"  Loss Function     : {self.cfg.LOSS.upper()}")
        print(f"  Image Size        : {self.cfg.IMG_SIZE}x{self.cfg.IMG_SIZE}")
        print("=" * 70)

    def plot_training_curves(self, history: Dict) -> None:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        axes[0].plot(history["epoch"], history["train_loss"],
                     label="Train Loss", color="#E74C3C")
        axes[0].set_title("Training Loss")
        axes[0].set_xlabel("Epoch")
        axes[0].set_ylabel("Loss")
        axes[0].legend()

        axes[1].plot(history["epoch"], history["val_auc"],
                     label="Val AUC", color="#2ECC71")
        axes[1].axhline(0.84, color="#E74C3C", linestyle="--",
                        lw=1.5, label="Target (0.84)")
        axes[1].set_title("Validation AUC")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("ROC-AUC")
        axes[1].legend()

        plt.suptitle("Training Curves — ClinicAI v2.0", fontweight="bold")
        plt.tight_layout()
        plt.savefig(CFG.OUTPUT_DIR / "training_curves.png", dpi=150)
        plt.close()
        print("Saved: training_curves.png")

<a id="main"></a>

## 🔧 Main Orchestrator — load_metadata() + main()

`main()` wires together all 20 sections in sequence:

```
load_metadata  →  EDA  →  patient split  →  transforms  →  datasets + loaders
  →  model  →  loss/optimiser/scheduler  →  Trainer.fit()
  →  load best checkpoint  →  full metrics  →  threshold optimisation
  →  GradCAM  →  error analysis  →  inference demo  →  final report
```

In [49]:
# ===========================================================================
# MAIN — Orchestrate the complete pipeline
# ===========================================================================

from torch.utils.data import DataLoader

def load_metadata(cfg: Config) -> pd.DataFrame:
    """Load and lightly preprocess the NIH ChestX-ray14 metadata CSV."""
    df = pd.read_csv(cfg.META_CSV)
    # Extract integer Patient ID from filename (e.g. "00000001_000.png" -> 1)
    df["Patient ID"] = df["Image Index"].apply(lambda x: int(x.split("_")[0]))
    print(f"Loaded {len(df):,} rows from {cfg.META_CSV.name}")
    return df

def build_dataloader(
    dataset,
    batch_size,
    shuffle,
    cfg
):

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=cfg.PIN_MEMORY,
        persistent_workers=(
            cfg.PERSISTENT_WORKERS and cfg.NUM_WORKERS > 0
        ),
        prefetch_factor=(
            cfg.PREFETCH_FACTOR if cfg.NUM_WORKERS > 0 else None
        ),
        drop_last=shuffle,
    )

def main():
    """End-to-end pipeline orchestration."""
    print("\n" + "=" * 60)
    print("  ClinicAI v2.1 — NIH ChestX-ray14 Pipeline")
    print("=" * 60 + "\n")

    # ── 1. Load metadata ─────────────────────────────────────────────
    df_raw = load_metadata(CFG)

    # ── 2. EDA  (Section 3) ──────────────────────────────────────────
    analyser = DataAnalyser(df_raw, CFG)
    df       = analyser.run()   # returns df with binary label columns

    # ── 3. Patient-Wise Split  (Section 4) ───────────────────────────
    splitter         = DataSplitter(df, CFG, val_ratio=0.15)
    train_df, val_df = splitter.split()

    # ── 4. Augmentation pipelines  (Section 5) ───────────────────────
    train_tfm = build_transforms("train", CFG)
    val_tfm   = build_transforms("val",   CFG)

    # ── 5. Datasets & DataLoaders  (Sections 5-6) ────────────────────
    train_ds   = ChestXray14Dataset(train_df, CFG.IMAGE_DIR, train_tfm, CFG)
    val_ds     = ChestXray14Dataset(val_df,   CFG.IMAGE_DIR, val_tfm,   CFG)
    pos_weight = train_ds.get_pos_weight()

    train_loader = build_dataloader(train_ds, CFG.BATCH_SIZE, shuffle=True,  cfg=CFG)
    val_loader   = build_dataloader(val_ds,   CFG.BATCH_SIZE, shuffle=False, cfg=CFG)
    print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

    # ── 6. Model  (Section 7) ────────────────────────────────────────
    model = ChestXrayModel(CFG).to(DEVICE)
    model = apply_performance_opts(model, CFG)
    print(f"Total params     : {model.param_count()/1e6:.1f} M")
    print(f"Trainable params : {model.trainable_param_count()/1e6:.1f} M")

    # ── 7. Loss / Optimiser / Scheduler  (Sections 8-10) ─────────────
    criterion = build_loss(CFG, pos_weight)
    optimiser = build_optimiser(model, CFG)
    scheduler = build_scheduler(optimiser, CFG)
    evaluator = Evaluator(CFG)

    # ── 8. Training  (Section 11) ────────────────────────────────────
    trainer = Trainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        criterion=criterion, optimiser=optimiser, scheduler=scheduler,
        cfg=CFG, evaluator=evaluator,
    )
    t0      = time.time()
    history = trainer.fit()
    t_train = time.time() - t0

    # ── 9. Load best EMA checkpoint ───────────────────────────────────
    best_ckpt = CFG.CHECKPOINT_DIR / "best_model.pth"
        
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["ema"])
        print(f"Loaded best EMA checkpoint (epoch {ckpt['epoch']})")
        
    model.eval()

    # ── 10. Full validation metrics  (Section 12) ────────────────────
    print("\nComputing full validation metrics...")
    val_probs, val_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE, non_blocking=True)
            with autocast(enabled=CFG.AMP):
                logits = model(images)
            val_probs.append(torch.sigmoid(logits).cpu().numpy())
            val_labels.append(labels.numpy())
    val_probs  = np.concatenate(val_probs,  axis=0)
    val_labels = np.concatenate(val_labels, axis=0)

    best_metrics = evaluator.compute(val_labels, val_probs)
    evaluator.plot_per_class_auc(best_metrics["per_class_auc"])
    evaluator.plot_roc_curves(val_labels, val_probs)
    y_pred_05 = (val_probs >= 0.5).astype(int)
    evaluator.plot_confusion_matrices(val_labels, y_pred_05)

    # ── 11. Threshold Optimisation  (Section 13) ─────────────────────
    thresh_opt = ThresholdOptimiser(CFG)
    thresholds = thresh_opt.fit(val_labels, val_probs)
    thresh_opt.save()

    best_metrics_opt = evaluator.compute(val_labels, val_probs, thresholds)
    print(f"\n  AUC  (fixed  0.5) : {best_metrics['auc_macro']:.4f}")
    print(f"  AUC  (opt thresh) : {best_metrics_opt['auc_macro']:.4f}")
    print(f"  F1   (fixed  0.5) : {best_metrics['f1']:.4f}")
    print(f"  F1   (opt thresh) : {best_metrics_opt['f1']:.4f}")

    # ── 12. Explainability  (Section 15) ─────────────────────────────
    if GRADCAM_AVAILABLE:
        cam_vis = GradCAMVisualiser(model, CFG)
        sample  = val_df.iloc[0]["Image Index"]
        cam_vis.visualise(CFG.IMAGE_DIR / sample,
                          class_indices=list(range(CFG.NUM_CLASSES)))

    # ── 13. Error Analysis  (Section 17) ─────────────────────────────
    err_analyser = ErrorAnalyser(val_df, CFG)
    err_analyser.analyse(val_labels, val_probs, thresholds, CFG.IMAGE_DIR, n=4)

    # ── 14. Inference Pipeline demo  (Section 16) ────────────────────
    pipeline   = InferencePipeline(model, CFG, thresholds)
    sample_img = CFG.IMAGE_DIR / val_df.iloc[0]["Image Index"]
    if sample_img.exists():
        result = pipeline.predict_single(sample_img, use_tta=True)
        print("\nSample TTA Prediction:")
        for cls, prob in result["probabilities"].items():
            flag = "[X]" if result["predictions"][cls] else "   "
            print(f"  {flag} {cls:<22} : {prob:.4f}")
        print(f"  Detected: {result['detected_labels']}")

    # ── 15. Final Report  (Section 20) ───────────────────────────────
    reporter = Reporter(CFG)
    reporter.print_report(history, best_metrics_opt, thresholds, model, t_train)
    reporter.plot_training_curves(history)

    # ── Cleanup ───────────────────────────────────────────────────────
    del model, trainer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    print("\nClinicAI v2.1 pipeline complete!")
    return best_metrics_opt

<a id="run"></a>

## ▶️ ▶ Run Full Pipeline

Execute the complete end-to-end training and evaluation pipeline.

> **Estimated time on Kaggle T4:** ~3–4 hours for 30 epochs  
> **Expected result:** Macro ROC-AUC > 0.84

Adjust `CFG.EPOCHS`, `CFG.BACKBONE`, or `CFG.LOSS` in Section 2 before running.

In [ ]:
# ── Run the full pipeline ───────────────────────────────────────────
if __name__ == "__main__":
    results = main()
else:
    results = main()   # always run in notebook context


  ClinicAI v2.1 — NIH ChestX-ray14 Pipeline

Loaded 112,120 rows from Data_Entry_2017.csv
Saved: class_distribution.png
Saved: cooccurrence_matrix.png
Saved: patient_distribution.png
Saved: view_position.png
Saved: sample_images.png

Class Imbalance Statistics
                    # Positive  # Negative  neg:pos ratio  Prevalence %
Infiltration             19894       92226       4.635870     17.743489
Effusion                 13317       98803       7.419314     11.877453
Atelectasis              11559      100561       8.699801     10.309490
Nodule                    6331      105789      16.709683      5.646629
Mass                      5782      106338      18.391214      5.156975
Pneumothorax              5302      106818      20.146737      4.728862
Consolidation             4667      107453      23.023998      4.162504
Pleural_Thickening        3385      108735      32.122600      3.019087
Cardiomegaly              2776      109344      39.389049      2.475919
Emphysema         

Epoch 1/30:   0%|          | 0/2981 [00:00<?, ?it/s]

In [ ]:
# 0. تسطيب المكتبة (لو مش متسطبة)
!pip install -q grad-cam

# ===========================================================================
# TESTING GRADCAM (FINAL WITH AUTO-IMAGE SEARCH)
# ===========================================================================

import torch
import cv2
import numpy as np
import math
import matplotlib.pyplot as plt
from pathlib import Path
from torch.cuda.amp import autocast
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# تعديل المتغير العام
global GRADCAM_AVAILABLE
GRADCAM_AVAILABLE = True

# 1. تعريف الموديل وتحميل الأوزان
model = ChestXrayModel(CFG).to(DEVICE)
checkpoint_path = Path("/kaggle/input/notebooks/agnefits/clinicai-v2-0/checkpoints/best_model.pth")

if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["ema"])
    print("✓ Successfully loaded pre-trained model weights!")
else:
    print(f"✗ Error: Checkpoint not found at {checkpoint_path}")

# 2. تشغيل الـ GradCAM المُحسن
model.eval()
cam_vis = GradCAMVisualiser(model, CFG)

# 3. البحث التلقائي عن الصورة عشان نتجنب خطأ المسار
print("Searching for the test image...")
image_filename = "00000001_000.png"
found_images = list(Path("/kaggle/input").rglob(image_filename))

if found_images:
    test_image_path = str(found_images[0])
    print(f"✓ Found image at: {test_image_path}")
else:
    # لو ملقاش الصورة دي، هيجيب أي صورة png من الداتا
    print(f"✗ Could not find {image_filename}. Looking for any X-ray image...")
    any_images = list(Path("/kaggle/input").rglob("*.png"))
    if any_images:
        test_image_path = str(any_images[0])
        print(f"✓ Testing with fallback image: {test_image_path}")
    else:
        test_image_path = None
        print("✗ Fatal Error: No .png images found in /kaggle/input!")

# 4. استدعاء دالة الرسم
if test_image_path:
    print("Generating Heatmaps... Please wait.")
    cam_vis.visualise(test_image_path, class_indices=list(range(CFG.NUM_CLASSES)))